Graph-Based Song Recommendations Using Spotify Audio Feature Vectors 

A graph-based recommender that moves from familiar songs (Louvain Modularity, Degree Centrality) to new communities by following natural transition paths (Betweenness Centrality, Shortest Path) in the graph

Dataset: https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

from neo4j import GraphDatabase

spotify_df = pd.read_csv("dataset.csv", index_col=0)

### EDA

In [2]:
print(spotify_df.shape)
print(spotify_df.dtypes)
spotify_df.head()

(114000, 20)
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
# handle missing data
spotify_df.isnull().sum().sort_values(ascending=False)

album_name          1
track_name          1
artists             1
track_id            0
speechiness         0
time_signature      0
tempo               0
valence             0
liveness            0
instrumentalness    0
acousticness        0
loudness            0
mode                0
key                 0
energy              0
danceability        0
explicit            0
duration_ms         0
popularity          0
track_genre         0
dtype: int64

In [4]:
spotify_df = spotify_df.dropna(subset=['album_name'])

In [5]:
# verify if track_id is a good unique identifier
trackid_dupes = spotify_df['track_id'].duplicated().sum()
print(trackid_dupes)

# investigate duplicates
trackid_dupes = spotify_df[spotify_df['track_id'].duplicated(keep=False)]
print(trackid_dupes.sort_values('track_id'))

24259
                      track_id            artists  \
15028   001APMDOl3qtx1526T11n1  Pink Sweat$;Kirby   
103211  001APMDOl3qtx1526T11n1  Pink Sweat$;Kirby   
85578   001YQlnDSduXd5LgBd66gT        Soda Stereo   
100420  001YQlnDSduXd5LgBd66gT        Soda Stereo   
91801   003vvx7Niy0yvhvHt4a68B        The Killers   
...                        ...                ...   
72679   7zv2vmZq8OjS54BxFzI2wM             Attila   
22326   7zv2vmZq8OjS54BxFzI2wM             Attila   
2004    7zwn1eykZtZ5LODrf7c0tS  The Neighbourhood   
3100    7zwn1eykZtZ5LODrf7c0tS  The Neighbourhood   
91401   7zwn1eykZtZ5LODrf7c0tS  The Neighbourhood   

                                             album_name  \
15028                                           New RnB   
103211                                          New RnB   
85578                          Soda Stereo (Remastered)   
100420                         Soda Stereo (Remastered)   
91801                                          Hot Fuss   
...

In [6]:
# remove track_id duplicates (same song across different album versions) keeping most popular one
spotify_df.sort_values(by='popularity', ascending=False, inplace=True) 
spotify_df.drop_duplicates(subset='track_id', inplace=True)

# remove 
spotify_df.drop_duplicates(subset=['track_name','artists'], inplace=True)
spotify_df.reset_index(drop=True, inplace=True)

spotify_df.shape

(81343, 20)

In [7]:
# explore relevant numeric audio feature vectors: danceability, energy, speechiness, acousticness, instrumentalness, valence, tempo
song_features = spotify_df[['danceability','energy','speechiness','acousticness','instrumentalness','valence', 'tempo']]
song_features.describe()

,danceability,energy,speechiness,acousticness,instrumentalness,valence,tempo
count,81343.000000,81343.000000,81343.000000,81343.000000,81343.000000,81343.000000,81343.000000
mean,0.559289,0.634911,0.088984,0.329697,0.184747,0.463311,122.134904
std,0.177738,0.258637,0.116619,0.339951,0.331606,0.263408,30.123005
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.446000,0.455000,0.036100,0.015900,0.000000,0.241000,99.377500
50%,0.573000,0.678000,0.049100,0.190000,0.000089,0.449000,122.028000
75%,0.690000,0.856500,0.087000,0.629000,0.153000,0.676000,140.124000
max,0.985000,1.000000,0.965000,0.996000,1.000000,0.995000,243.372000


### Correlation heatmap

In [8]:
# correlation heatmap
corr = song_features.corr().style.background_gradient(cmap='Blues')
corr

,danceability,energy,speechiness,acousticness,instrumentalness,valence,tempo
danceability,1.000000,0.134697,0.108365,-0.167839,-0.191638,0.491963,-0.015096
energy,0.134697,1.000000,0.139930,-0.731064,-0.186146,0.253539,0.260598
speechiness,0.108365,0.139930,1.000000,0.012264,-0.107059,0.035096,-0.003374
acousticness,-0.167839,-0.731064,0.012264,1.000000,0.095007,-0.101561,-0.223376
instrumentalness,-0.191638,-0.186146,-0.107059,0.095007,1.000000,-0.332371,-0.058773
valence,0.491963,0.253539,0.035096,-0.101561,-0.332371,1.000000,0.093250
tempo,-0.015096,0.260598,-0.003374,-0.223376,-0.058773,0.093250,1.000000


In [9]:
# standardize features to same scale
scaler = StandardScaler()
spotify_scaled = scaler.fit_transform(song_features)

# validate scaled values
spotify_scaled_df = pd.DataFrame(spotify_scaled, columns=song_features.columns)
spotify_scaled_df.head()

,danceability,energy,speechiness,acousticness,instrumentalness,valence,tempo
0,0.870451,-0.629886,-0.022162,-0.931603,-0.557117,-0.855374,0.298315
1,0.347205,0.568714,-0.385742,-0.933074,-0.457615,0.329110,0.195802
2,1.551233,0.170470,-0.450912,0.745120,-0.557125,1.468036,0.094450
3,0.009627,1.276274,-0.468919,-0.958578,-0.557110,-0.604810,0.196034
4,0.510368,0.309662,1.406434,-0.677741,-0.556254,-1.048991,-0.513329


### Genre similarity analysis

In [ ]:
# Select the audio feature columns for genre analysis
genre_features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Group by genre and get the mean of these features
genre_means = spotify_df.groupby('track_genre')[genre_features].mean()

# standardize genre profiles
scaler = StandardScaler()
genre_means_scaled = scaler.fit_transform(genre_means)

# compute cosine similarity between genre profiles
similarity_matrix = cosine_similarity(genre_means_scaled)
similarity_df = pd.DataFrame(similarity_matrix, index=genre_means.index, columns=genre_means.index)

similarity_df.head()

track_genre,acoustic,afrobeat,alt-rock,alternative,ambient,anime,black-metal,bluegrass,blues,brazil,...,spanish,study,swedish,synth-pop,tango,techno,trance,trip-hop,turkish,world-music
track_genre,,,,,,,,,,,,,,,,,,,,,
acoustic,1.000000,-0.280143,-0.430991,-0.184975,0.570020,-0.150529,-0.415338,0.450975,0.376841,-0.042728,...,-0.351757,0.217618,0.227485,-0.286997,0.706710,-0.417012,-0.629122,-0.173537,0.271932,0.334291
afrobeat,-0.280143,1.000000,0.279385,0.245120,-0.365728,-0.170299,-0.527524,0.187537,0.547222,-0.184240,...,0.391874,0.204192,0.481266,0.631303,0.021132,0.101889,-0.243781,0.488232,0.147716,-0.884927
alt-rock,-0.430991,0.279385,1.000000,0.762513,-0.753278,-0.277437,0.167989,-0.169004,0.417235,-0.084773,...,0.712836,-0.709302,0.566251,0.810342,-0.640016,0.043963,0.193420,-0.222355,0.046206,-0.199120
alternative,-0.184975,0.245120,0.762513,1.000000,-0.671322,-0.381491,-0.098335,-0.467153,0.551369,0.207739,...,0.706466,-0.504322,0.572566,0.633417,-0.468501,-0.070834,-0.079266,-0.137203,0.483746,-0.182365
ambient,0.570020,-0.365728,-0.753278,-0.671322,1.000000,0.544671,0.137422,0.200576,-0.226136,-0.351845,...,-0.943847,0.676836,-0.564942,-0.599944,0.516342,0.043812,-0.133769,0.320010,-0.301993,0.519974


In [ ]:
plt.figure(figsize=(20, 20))
sns.clustermap(similarity_df, cmap='viridis', figsize=(15, 15))
plt.title('Similarity Cluster Map of Spotify Genres')
plt.show()

In [ ]:
print('Least similar to show-tunes:')
print(similarity_df['show-tunes'].sort_values(ascending=True).head(6))

Least similar to show-tunes:
track_genre
breakbeat           -0.938207
party               -0.765994
progressive-house   -0.761683
industrial          -0.750203
j-idol              -0.721864
chicago-house       -0.716238
Name: show-tunes, dtype: float64


In [ ]:
print('Most similar to show-tunes:')
print(similarity_df['show-tunes'].sort_values(ascending=False).head(6))

Most similar to show-tunes:
track_genre
show-tunes    1.000000
romance       0.957365
opera         0.905172
jazz          0.871246
disney        0.795873
tango         0.774387
Name: show-tunes, dtype: float64


# Build Initial Graph

In [ ]:
# neo4j helper functions

# define neo4j connection
driver = GraphDatabase.driver(
    'bolt://localhost:7687',
    auth=('neo4j','205spotify'))

session = driver.session(database="neo4j")

# convert results to pandas dataframe
def my_neo4j_run_query_pandas(query, **kwargs):
    with driver.session(database="neo4j") as session:
        result = session.run(query, **kwargs)
        df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    return df

# run neo4j query
def neo4j_run(query, **kwargs):
    with driver.session(database="neo4j") as session:
        return session.run(query, **kwargs)

# wipe neo4j graph
# wiping in batches of 5k rows prevents memorypooloutofmemory error when trying to wipe database
def neo4j_batch_wipe():
    with driver.session(database="neo4j") as session:
        batch_wipe_query = """
        MATCH (n)
        CALL {
            WITH n
            DETACH DELETE n
        } IN TRANSACTIONS OF 5000 ROWS
        """
        session.run(batch_wipe_query)

In [ ]:
# Build initial Neo4j graph
node_query = """
    UNWIND $rows as row
    MERGE (s:Song {id: row.track_id})
    SET s.track_name = row.track_name,
        s.genre = row.track_genre,
        s.artists = row.artists,
        s.album = row.album_name,
        s.popularity = row.popularity
"""

# convert song data to list (neo4j needs this format)
node_list = spotify_df.to_dict('records')

print('Wiping database')
neo4j_batch_wipe()
    
print('Create nodes')
session.run(node_query, rows=node_list)

# Create edges using KNN (no genre restriction)
print('Running KNN')

# match tracks to five nearest neighbors using cosine similarity
knn = NearestNeighbors(n_neighbors=6, metric='cosine')
knn.fit(spotify_scaled)

# get indices of nearest neighbors and their distances to the track located at the corresponding index
distances, indices = knn.kneighbors(spotify_scaled)

# create list of edges pointing to most similar tracks
edges = []

for i in range(len(spotify_df)):  
    src_id = spotify_df.iloc[i]['track_id']
    
    for j in range(1, 6):  
        nbr_idx = indices[i][j]  
        nbr_id = spotify_df.iloc[nbr_idx]['track_id']  
        cos_dist = distances[i][j]  

        # add data to list of edges
        edges.append({
            'source_id': src_id,
            'target_id': nbr_id,
            'distance': float(cos_dist),
            'similarity': float(1 - cos_dist)
        })

# convert edge data to dataframe
edges_df = pd.DataFrame(edges)

# convert dataframe of edges to list for neo4j
edges_list = edges_df.to_dict('records')

# run edge query
edge_query = """
    UNWIND $rows AS row
    MATCH (source:Song {id: row.source_id})
    MATCH (target:Song {id: row.target_id})
    MERGE (source)-[r:SIMILAR_TO]->(target)
    SET r.similarity = row.similarity,
        r.distance   = row.distance,
        r.cost = row.distance
"""

batch_size = 5000
for start in range(0, len(edges_list), batch_size):
    batch = edges_list[start:start + batch_size]
    neo4j_run(edge_query, rows=batch)

Wiping database


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (n) { ... }', position=<SummaryInputPosition line=3, column=9, offset=27>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 27, 'line': 3, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (n)\n        CALL {\n            with n\n            detach delete n\n        } in transactions of 5000 rows\n        '


Create nodes
Running KNN


# Graph Projection

In [ ]:
# CREATE GRAPH PROJECTION

# standardize relationship properties
sr_query = """
MATCH ()-[r:SIMILAR_TO]->()
SET r.similarity = r.similarity,
    r.cost = r.distance
"""
neo4j_run(sr_query)

# drop previous in memory graph projection
drop_proj_query = """
CALL gds.graph.exists('song_graph') YIELD exists
WITH exists
CALL {
    WITH exists
    WHERE exists
    CALL gds.graph.drop('song_graph', false) YIELD graphName
    RETURN graphName
}
RETURN coalesce(graphName, 'not_found') AS graphName
"""
neo4j_run(drop_proj_query)

# project graph with both edge types
new_proj_query = """
CALL gds.graph.project(
    'song_graph',
    'Song',
    {
        SIMILAR_TO: {
            orientation: 'UNDIRECTED',
            properties: ['similarity', 'cost']
        }
    }
)
"""
neo4j_run(new_proj_query)



# Louvain

In [ ]:
# run louvain for communityID
louvain_query = """
CALL gds.louvain.stream(
    'song_graph',
    {
        relationshipWeightProperty: 'similarity'
    }
)
YIELD nodeId, communityId
RETURN
    gds.util.asNode(nodeId).id AS track_id,
    gds.util.asNode(nodeId).track_name AS track_name,
    gds.util.asNode(nodeId).genre AS genre,
    communityId AS community
ORDER BY community, track_name
"""

# write communityIDs back to songs in neo4j
# this will assist pathing process later
lou_write_query = """
CALL gds.louvain.write(
    'song_graph',
    {
        relationshipWeightProperty: 'similarity',
        writeProperty: 'community'
    }
)
YIELD communityCount, modularity
RETURN communityCount, modularity
"""

louvain_write = my_neo4j_run_query_pandas(lou_write_query)
print('')
print(louvain_write)

# store louvain results to assist in pathing
louvain_df = my_neo4j_run_query_pandas("""
    MATCH (s:Song)
    RETURN s.id AS track_id,
           s.track_name AS track_name,
           s.genre AS genre,
           s.community AS community
    ORDER BY community, track_name
""")

print('Total Communities:', louvain_df['community'].nunique())
print('')
print(louvain_df.head())



   communityCount  modularity
0              39    0.878878
Total Communities: 39

                 track_id                          track_name       genre  \
0  3KKk48f33mlB56F5L5nbJk  "Don Carlos" Roderigo'S Death Aria     romance   
1  5OiONTndVC5YOMXg6VC5xs    "Eugene Onegin" Ariozo Of Onegin     romance   
2  5ezuHIXXlsPQY4rbsSKT1W                       "He Was Tall"  show-tunes   
3  280zqg7gblMTRU1Hc5VxpY             "Loneliness Of Evening"  show-tunes   
4  6ua4ZcCoSABREsczFvGLI6                  'O Sole Mio - Live       opera   

   community  
0       4252  
1       4252  
2       4252  
3       4252  
4       4252  


## Degree Centrality

In [ ]:
# DEGREE CENTRALITY
# run degree centrality
degree_query = """
CALL gds.degree.stream('song_graph')
YIELD nodeId, score
RETURN
    gds.util.asNode(nodeId).id AS track_id,
    score AS degree
ORDER BY degree DESC
"""

degree_df = my_neo4j_run_query_pandas(degree_query)

print(degree_df.head())
print('')

# write degree scores back to nodes in neo4j
degree_write_query = """
CALL gds.degree.write(
    'song_graph',
    {
        writeProperty: 'degree'
    }
)
YIELD nodePropertiesWritten
RETURN nodePropertiesWritten
"""

degree_write = my_neo4j_run_query_pandas(degree_write_query)
print(degree_write)

# merge degree to louvain_df to build set of relevant features
track_feats = louvain_df.merge(
    degree_df[['track_id', 'degree']],
    on='track_id',
    how='left'
)
track_feats['degree'] = track_feats['degree'].fillna(0)

# merge popularity back to features to choose most popular song first
track_feats = track_feats.merge(
    spotify_df[['track_id', 'popularity']],
    on='track_id',
    how='left'
)

                 track_id   betweenness
0  36YlzbkAZLnOke9yib6Yin  68628.543564
1  0oiXA5d3B3QAbHFwom7OJZ  55722.371948
2  395WRTyNwMGx80cDXqg6vM  51580.578798
3  3FSF2F62qs9Y5Ak65nqGsx  43903.079588
4  3mS1DMR2N8eQ0qGezDPTOx  43226.009454

   nodePropertiesWritten
0                  81343


## Betweenness Centrality

In [ ]:
# run approximate betweenness centrality (UNWEIGHTED)
bw_query = """
CALL gds.betweenness.stream(
    'song_graph',
    {
        samplingSize: 100,
        samplingSeed: 0
    }
)
YIELD nodeId, score
RETURN
    gds.util.asNode(nodeId).id AS track_id,
    score AS betweenness
ORDER BY betweenness DESC
"""

betweenness_df = my_neo4j_run_query_pandas(
    bw_query,
    sampling_size=100,
    sampling_seed=0
)

print(betweenness_df.head())
print('')

# write betweenness scores back to nodes in neo4j
bw_write_query = """
CALL gds.betweenness.write(
    'song_graph',
    {
        samplingSize: 100,
        samplingSeed: 0,
        writeProperty: 'betweenness'
    }
)
YIELD nodePropertiesWritten
RETURN nodePropertiesWritten
"""

between_write = my_neo4j_run_query_pandas(bw_write_query)
print(between_write)

# merge betweenness into existing track features
track_feats = track_feats.merge(
    betweenness_df[['track_id', 'betweenness']],
    on='track_id',
    how='left'
)
track_feats['betweenness'] = track_feats['betweenness'].fillna(0)

                 track_id   betweenness
0  0oiXA5d3B3QAbHFwom7OJZ  92955.649434
1  3FSF2F62qs9Y5Ak65nqGsx  77373.093595
2  2kkvB3RNRzwjFdGhaUA0tz  73212.965358
3  4O5xErTIU7SLQ6QJ6SUc0D  62884.592812
4  4TZh9HGsgaH3qzX40TPsF3  62684.789205

   nodePropertiesWritten
0                  81343


In [ ]:
top_bw = betweenness_df.head(11) 
top_bw.merge(
    spotify_df[['track_id', 'track_name', 'artists']], 
    how='inner', 
    on='track_id'
)[['track_name', 'artists', 'betweenness']]

,track_name,artists,betweenness
0,By My Side,Massane;MAGNUS,92955.649434
1,Angha,Mehdi Khayami;Lorenzo Gorli,77373.093595
2,Layla,Derek & The Dominos,73212.965358
3,Ocean Sounds: Relaxing Wind - Loopable,The Ocean Waves Sounds,62884.592812
4,Wenn Baby nicht schläft,Weißes Rauschen HD,62684.789205
5,"Tzigane, M. 76 (Version for Violin & Orchestra...",Maurice Ravel;Luis Perez Canabal;Central Washi...,62564.242524
6,Čtyři Slunce,Vypsana Fixa,61829.527186
7,Twenty Two,Farzad Golpayegani,59186.858033
8,Dom fyra årstiderna,Håkan Hellström,56278.093893
9,Waves To Sleep,Ocean Waves For Sleep,56095.302414


## Identify Target Community
Calculate centroids of each community
Find community whose centroid is the closest to the input/source community and set it as target

In [ ]:
def find_target_community(input_community, track_feats, spotify_scaled, spotify_df):
    # dictionary mapping track_id to its index in initial spotify data
    map_id_to_idx = {row['track_id']: i for i, row in spotify_df.iterrows()}

    community_centroids = {}
    

    for community, group in track_feats.groupby('community'):  
        # get the index for every track in this community and drop duplicates
        idxs = [map_id_to_idx[tid] for tid in group['track_id'] if tid in map_id_to_idx]
        
        # get only songs in this community
        community_centroids[community] = spotify_scaled[idxs].mean(axis=0)

    # get centroid for input/source song's community
    input_centroid = community_centroids[input_community]

    # initialize variables to track closest community search
    best_community = None
    best_distance  = float('inf')

    # loop through every community and compare its centroid to input/source community's centroid
    for community, centroid in community_centroids.items():
        if community == input_community:
            continue
        
        # calculate euclidean distance between centroids
        dist = np.linalg.norm(input_centroid - centroid)
        if dist < best_distance:  
            best_distance = dist 
            best_community = community  

    return best_community


## Find Bridge Song

In [ ]:
def find_bridge_song(input_community, target_community):
    # query orders results by descending betweenness and filters for top result
    # also filters for tracks where edges are in both input and target community
    bridge_query = """
        MATCH (bridge:Song)-[:SIMILAR_TO]-(a:Song),
              (bridge:Song)-[:SIMILAR_TO]-(b:Song)
        WHERE a.community = $input_community
          AND b.community = $target_community
          AND bridge.community <> $input_community
          AND bridge.betweenness IS NOT NULL
        RETURN DISTINCT
            bridge.id AS track_id,
            bridge.track_name AS track_name,
            bridge.genre AS genre,
            bridge.community AS community,
            bridge.betweenness AS betweenness
        ORDER BY betweenness DESC
        LIMIT 1
    """

    result = my_neo4j_run_query_pandas(
        bridge_query,
        input_community=int(input_community),
        target_community=int(target_community)
    )

    if result.empty:
        return None, None

    best = result.iloc[0]
    return best['track_id'], int(best['community'])

## Decide on destination

In [ ]:
# pick a destination track
def pick_destination(target_comm, bridge_id):
    target_query = """
    MATCH (s:Song)
    WHERE s.community = $target_community
    AND s.id <> $bridge_track_id
    RETURN 
        s.id AS track_id, s.track_name AS track_name,
        s.genre AS genre, s.popularity AS popularity
    ORDER BY popularity DESC
    LIMIT 20
    """

    target_output = my_neo4j_run_query_pandas(
        target_query,
        target_community=int(target_comm),
        bridge_track_id=bridge_id
    )

    rand_target_idxs = np.random.default_rng(0)

    target_track = target_output.iloc[rand_target_idxs.integers(0, len(target_output))]

    return target_track

# Dijkstra (Shortest Path)

In [ ]:
# define Dijkstra helper function to find shortest path by segmenting into two paths:
# path 1: shortest input to bridge
# path 2: shortest bridge to target

def shortest_path(start_track_id, target_track_id):
    
    query = """
    MATCH (source:Song {id: $source_id}), (target:Song {id: $target_id})
    CALL gds.shortestPath.dijkstra.stream('song_graph',
        {
            sourceNode: source,
            targetNode: target,
            relationshipWeightProperty: 'cost'
        }
    )
    YIELD totalCost, nodeIds
    RETURN
        totalCost,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).id] AS track_ids,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).track_name] AS track_names,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).artists] AS artists,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).genre] AS genres,
        [nodeId IN nodeIds | gds.util.asNode(nodeId).community]  AS communities
    """

    path_result = my_neo4j_run_query_pandas(
        query,
        source_id=start_track_id,
        target_id=target_track_id
    )
    #print(path_result)

    path = path_result.iloc[0]

    path_df = pd.DataFrame({
        'track_id': path['track_ids'],
        'track_name': path['track_names'],
        'artist': path['artists'],
        'genre': path['genres'],
        'community': path['communities']
    })

    path_df['total_cost'] = path['totalCost']

    return path_df

In [ ]:
def generate_queue(input_track_name):

    # search track data for songs that are a match to the input song
    # sort to get most popular song 
    matches = track_feats[
        track_feats['track_name'].str.lower().str.contains(input_track_name.lower(), na=False)
    ].sort_values('popularity', ascending=False)

    if matches.empty:
        raise ValueError(f"No matches found for '{input_track_name}'")

    # select most popular song
    input_song = matches.iloc[0]
    
    # get track id to use to identify node in neo4j
    input_id = input_song['track_id']
    
    # get louvain community that input song belongs to
    input_community = int(input_song['community'])

    # pick a few familiar songs from the same community using degree centrality
    familiar_songs = track_feats[
        (track_feats['community'] == input_community) &
        (track_feats['track_id'] != input_id)
    ].sort_values(['degree', 'popularity'], ascending=False).head(3)

    familiar_ids = [input_id] + familiar_songs['track_id'].tolist()

    # run target community function 
    target_community = find_target_community(
        input_community, track_feats, spotify_scaled, spotify_df
    )

    # run bridge song finder function
    bridge_id, bridge_community = find_bridge_song(input_community, target_community)

    if bridge_id is None:
        raise ValueError("No bridge song found between input and target communities.")
    
    # pick a destination/target song in target community
    dest_row = pick_destination(target_community, bridge_id)
    
    # determine shortest path
    path1 = shortest_path(familiar_ids[-1], bridge_id)  # last familiar song to bridge
    path2 = shortest_path(bridge_id, dest_row['track_id'])  # bridge to target

    # build dataframe for familiar songs in the chosen order
    familiar_path = track_feats[
        track_feats['track_id'].isin(familiar_ids)
    ][['track_id', 'track_name', 'genre', 'community']].copy()

    familiar_path = familiar_path.set_index('track_id').loc[familiar_ids].reset_index()
    familiar_path['artist'] = familiar_path['track_id'].map(
        spotify_df.set_index('track_id')['artists']
    )
    familiar_path['total_cost'] = 0

    # combine familiar songs + shortest paths
    final_path = pd.concat(
        [familiar_path, path1.iloc[1:], path2.iloc[1:]],
        ignore_index=True
    )
    
    # prevent queue from playing same song more than once
    final_path = final_path.drop_duplicates(subset='track_id', keep='first')
    final_path = final_path.drop_duplicates(subset='track_name', keep='first')
    final_path = final_path.reset_index(drop=True)

    return final_path

In [ ]:
# run function to generate queue

# define starting track
input_track_name = 'bohemian rhapsody'

generate_queue(input_track_name)

,track_id,track_name,artist,genre,community,total_cost
0,4u7EnebtmKWzUH433cf5Qv,Bohemian Rhapsody - Remastered 2011,Queen,rock,57363,0.120060
1,5Rm70bhYNH2eTtHjlwgPY0,The Passion (Live Acoustic) - Bonus,Hillsong Worship;Brooke Ligertwood,world-music,57363,0.120060
2,2PHJSiIQQgVAuHqSxEp6F9,軌跡,Jay Chou,mandopop,57363,0.120060
3,0M8g4Bt0R12Y3U9MM9oXaT,Mary,Oingo Boingo,synth-pop,57363,0.120060
4,1KHdq8NK9QxnGjdXb55NiG,Falling in Love at a Coffee Shop,Landon Pigg,acoustic,57363,0.120060
5,5wTJw7Eu5mi0AqSYyIZLQs,O Rosto de Cristo,Sarah Farias,gospel,57363,0.120060
6,3RIYWuMEasEhXVNx722ZOC,Champagne Supernova,OneRepublic,piano,57363,0.120060
7,58AS4weYlUWvGztZbKIuAU,Leave Out All The Rest,Linkin Park,alternative,57363,0.120060
8,3j6s1vr84KzKlpF0QfHzDy,Army,Besomorph;Arcando;Neoni,house,57363,0.120060
9,4YXKVaWzHaVyDq2pVS5cgR,Proud,Marshmello,progressive-house,57363,0.120060


Note: to see edges in Neo4j browser, run:

MATCH (s:Song)-[r]->(n:Song) <br>
RETURN s, r, n